In [ ]:
# should be 0.8.6
import trl
print(trl.__version__)

In [ ]:
# FINE-TUNING WITH MISTRAL-7B + LoRA

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer
import json
from pathlib import Path


print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
def predicate_to_verb(predicate):
    mapping = {
        "hasNutrient": "contains",
        "usesTechnique": "is prepared using",
        "hasGuideline": "follows the guideline",
        "recommendsTechnique": "recommends",
        "aimsToImprove": "aims to improve",
        "affectsRiskOf": "affects the risk of",
        "associatedWithOutcome": "is associated with",
        "hasEnvironmentalImpact": "has an environmental impact",
        "guidelineTargetsImpact": "targets environmental impact",
        "affectsImpactCategory": "affects the environmental impact category"
    }
    return mapping.get(predicate, predicate)

In [ ]:
# define templates for different types of questions
TEMPLATES = {
    "hasNutrient": {
        "factoid": [
            "What nutrient is found in {s}?",
            "Which nutrient does {s} provide?",
            "What is the main nutrient in {s}?",
            "What does {s} contain that supports health?",
            "{s} is rich in which nutrient?"
        ],
        "list": [
            "List foods rich in {o}.",
            "Which ingredients are good sources of {o}?",
            "Give examples of foods containing {o}.",
            "Name three foods that are high in {o}."
        ],
        "constraint": [
            "Give two ingredients that contain {o}.",
            "Find vegan foods rich in {o}.",
            "List foods with high {o} content under 200 kcal per 100g."
        ]
    },
    "associatedWithOutcome": {
        "factoid": [
            "What health outcome is {s} associated with?",
            "How does consuming {s} affect health?",
            "{s} is linked to which health condition?",
            "What condition may be influenced by {s}?"
        ],
        "list": [
            "List foods associated with reduced risk of {o}.",
            "Which foods promote {o}?",
            "Name foods linked to {o}."
        ]
    },
    "affectsRiskOf": {
        "factoid": [
            "What health outcome does {s} influence?",
            "Which disease risk is affected by {s}?",
            "How does {s} affect the risk of {o}?",
            "What role does {s} play in preventing {o}?"
        ]
    },
    "usesTechnique": {
        "factoid": [
            "What technique is used to prepare {s}?",
            "How is {s} typically cooked or processed?",
            "What preparation method applies to {s}?"
        ]
    },
    "affectsImpactCategory": {
        "factoid": [
            "What environmental impact is influenced by {s}?",
            "Which sustainability impact is affected by {s}?",
            "How does {s} affect environmental footprint?"
        ],
        "reasoning": [
            "If a recipe uses {s}, what environmental impact may increase?",
            "When {s} is used, which environmental category might be affected?"
        ]
    },
    "hasGuideline": {
        "factoid": [
            "What dietary guideline applies to {s}?",
            "What recommendation is given regarding {s}?",
            "What official advice mentions {s}?"
        ]
    },
    "aimsToImprove": {
        "factoid": [
            "What health outcome does {s} aim to improve?",
            "What benefit does {s} target?",
            "Which condition is addressed by {s}?"
        ]
    },
    "guidelineTargetsImpact": {
        "factoid": [
            "What environmental impact does {s} aim to reduce?",
            "Which sustainability metric is targeted by {s}?"
        ]
    }
}

In [ ]:
# Load train, validation, test splits
train_data = load_dataset('json', data_files='data/train/test/val/train_instructions.jsonl', split='train')
val_data = load_dataset('json', data_files='data/train/test/val/val_instructions.jsonl', split='train')
test_data = load_dataset('json', data_files='data/train/test/val/test_instructions.jsonl', split='train')

print(f"Train samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples: {len(test_data)}")
print(f"\nSample instruction:")
print(train_data[0])

In [ ]:
# MISTRAL-7B WITH 4-BIT QUANTIZATION loading

train_value = 2 # to store multiple trainings

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-bit quantization config for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding=False,    
        truncation=True,
        max_length=256
        #add_special_tokens=False
    )

print(f"Model loaded: {model_name}")
print(f"Model size: ~7B parameters")
print(f"Memory footprint: ~7GB with 4-bit quantization")

In [ ]:
def format_instruction(sample, pred_conversion=True):
    """
    Format using tokenizer's chat template for consistency.
    """
    instruction = sample['instruction']
    input_text = sample.get('input', '').strip()
    output = sample['output']

    if pred_conversion:
        for pred in TEMPLATES.keys():
            if pred in output:
                output = output.replace(pred, predicate_to_verb(pred))

    # Build the prompt
    if input_text:
        user_message = f"{instruction}\n\nContext: {input_text}"
    else:
        user_message = instruction

    # Use chat template
    messages = [
        {"role": "system", "content": "You are a helpful assistant that answers questions about food and health relationships based on scientific evidence. Always cite your sources with page numbers."},
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": output}
    ]

    # Apply chat template
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    formatted = formatted.lstrip(tokenizer.bos_token)

    return {"text": formatted}

In [ ]:
train_dataset = train_data.map(format_instruction, remove_columns=train_data.column_names)
val_dataset = val_data.map(format_instruction, remove_columns=val_data.column_names)
test_dataset = test_data.map(format_instruction, remove_columns=test_data.column_names)
print(f"Formatted datasets ready:")
print(f"Train: {len(train_dataset)} samples")
print(f"Val: {len(val_dataset)} samples")
print(f"Test: {len(test_dataset)} samples")
print(f"sample:\n{train_dataset[0]}")

In [ ]:
import transformers, accelerate, torch
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Torch:", torch.__version__)

In [ ]:
import bitsandbytes as bnb
print("BitsAndBytes:", bnb.__version__)


In [ ]:
# CONFIGURE LoRA (Low-Rank Adaptation) Configuration
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128,garbage_collection_threshold:0.6"


lora_config = LoraConfig(
    r=8, #16                        # Rank of LoRA matrices (higher = more capacity)
    lora_alpha=16, #32              # Scaling factor (typically 2*r)
    target_modules=[                # Which layers to apply LoRA to
        "q_proj",
        #"k_proj",
        "v_proj",
        #"o_proj",
        #"gate_proj",
        #"up_proj",
        #"down_proj",
    ],
    lora_dropout=0.05,           # Dropout for LoRA layers
    bias="none",                 # Don't train bias parameters
    task_type="CAUSAL_LM"        # Task type
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
trainable_percent = 100 * trainable_params / all_params

print(f"Trainable parameters: {trainable_params:,} / {all_params:,} ({trainable_percent:.2f}%)")
print(f"Memory efficient: Only training {trainable_percent:.2f}% of parameters!")

In [ ]:
# TRAINING ARGUMENTS CONFIGURATION
if not getattr(model, "is_gradient_checkpointing", False):
    try:
        model.gradient_checkpointing_enable()
    except Exception:
        # fallback for PEFT-wrapped models
        for m in model.modules():
            if hasattr(m, "gradient_checkpointing"):
                m.gradient_checkpointing = True

output_dir = f"models/fine_tuned_model/train{train_value}"
logging_dir = f"models/fine_tuned_model/train{train_value}/logs"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=20,                   # Number of training epochs
    per_device_train_batch_size=4,         # Batch size per GPU/CPU
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,         # Accumulate gradients (effective batch = 16)
    learning_rate=2e-4,                    # Learning rate for AdamW
    lr_scheduler_type="cosine",            # Learning rate schedule
    warmup_ratio=0.03,                     # Warmup 3% of total steps

    # Logging and evaluation
    logging_dir=logging_dir,
    logging_steps=50,                      # Log every 50 steps
    logging_strategy="steps",
    eval_strategy="epoch",                 # Evaluate during training
    #eval_steps=500,                       
    save_strategy="epoch",                 # Save checkpoints
    #save_steps=500,                       
    save_total_limit=2,                    # Keep only 2 best checkpoints

    # Early stopping and best model tracking
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,               # Lower loss is better

    # Optimization
    fp16=True,                             # Use mixed precision (faster)
    optim="paged_adamw_8bit",              # Memory-efficient optimizer
    #max_grad_norm=1.0,                    # Gradient clipping

    # Other settings
    report_to=None,               
    #push_to_hub=False,                    
    remove_unused_columns=False,

    torch_compile=False,
    gradient_checkpointing=True,
)

print("Training arguments configured")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Total training steps: ~{len(train_dataset) * training_args.num_train_epochs // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")

In [ ]:
print("Tokenizing datasets...")
tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_val   = val_dataset.map(tokenize_function, batched=True, remove_columns=val_dataset.column_names)
tokenized_test  = test_dataset.map(tokenize_function, batched=True, remove_columns=test_dataset.column_names)

In [ ]:
from transformers import DataCollatorWithPadding

# Base collator pads input_ids/attention_mask to the longest in the batch
base_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest", return_tensors="pt")

def data_collator(features):
    """
    features: list[dict] where each dict contains 'input_ids' and 'attention_mask' (and maybe 'labels' if present).
    We:
      - pad the batch to the longest sample
      - create labels from input_ids and mask padding positions with -100
    """
    batch = base_collator(features)  # returns tensors: input_ids, attention_mask, etc.
    # Create labels from input_ids and mask padding tokens (attention_mask==0) to -100 for loss ignoring
    labels = batch["input_ids"].clone()
    # positions where attention_mask == 0 are padding -> set to -100 so loss ignores them
    labels[batch["attention_mask"] == 0] = -100
    batch["labels"] = labels
    return batch

In [ ]:
tokenized_dataset = tokenized_train
print(f"\nDataset size: {len(tokenized_dataset)}")
print(f"Dataset columns: {tokenized_dataset.column_names}")

# Look at first example
print("\n" + "-"*60)
print("First example:")
print("-"*60)
sample = tokenized_dataset[0]
print(f"Keys: {sample.keys()}")
print(f"\nInput IDs shape: {len(sample['input_ids'])}")
print(f"Input IDs (first 20): {sample['input_ids'][30:50]}")
print(f"\nAttention mask (first 20): {sample['attention_mask'][30:50]}")

# Decode to see the actual text
print("\n" + "-"*60)
print("Decoded text:")
print("-"*60)
decoded = tokenizer.decode(sample['input_ids'], skip_special_tokens=False)
print(decoded)

In [ ]:
# TRAINER INITIALIZE
from trl import SFTTrainer

max_seq_length = 256

def formatting_func(example):
    return example["text"]



trainer = SFTTrainer(
    model=model,
    data_collator=data_collator,
    train_dataset=tokenized_train, #train_dataset,
    eval_dataset=tokenized_val, #val_dataset,
    peft_config=lora_config,
    tokenizer=tokenizer,
    args=training_args,
    formatting_func=formatting_func,
    max_seq_length=max_seq_length,
    packing=False,
)

torch.cuda.empty_cache()
print("GPU memory before train:", torch.cuda.memory_reserved(0) / 1024**2, "MiB")


print(f"Training on {len(tokenized_train)} samples")
print(f"Validating on {len(tokenized_val)} samples")

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
# take a tiny batch of examples from tokenized_train (first 4)
examples = [tokenized_train[i] for i in range(min(4, len(tokenized_train)))]
batch = data_collator(examples)

print("input_ids shape:", batch["input_ids"].shape)
print("attention_mask shape:", batch["attention_mask"].shape)
print("labels shape:", batch["labels"].shape)
# show that there are -100 entries in labels where attention_mask==0
print("num padded label positions (should match padding positions):", (batch["labels"] == -100).sum().item())

In [ ]:
# Access the processed dataset
print("\n" + "="*60)
print("INSPECTING TOKENIZED DATASET")
print("="*60)

# Get the train dataset (already tokenized by SFTTrainer)
tokenized_dataset = trainer.train_dataset

print(f"\nDataset size: {len(tokenized_dataset)}")
print(f"Dataset columns: {tokenized_dataset.column_names}")

# Look at first example
print("\n" + "-"*60)
print("First example:")
print("-"*60)
sample = tokenized_dataset[0]
print(f"Keys: {sample.keys()}")
print(f"\nInput IDs shape: {len(sample['input_ids'])}")
print(f"Input IDs (first 20): {sample['input_ids'][30:50]}")
print(f"\nAttention mask (first 20): {sample['attention_mask'][30:50]}")

# Decode to see the actual text
print("\n" + "-"*60)
print("Decoded text:")
print("-"*60)
decoded = tokenizer.decode(sample['input_ids'], skip_special_tokens=False)
print(decoded)

print("\n" + "-"*60)
print("Single tokens:")
print("-"*60)
for i in range(len(sample['input_ids'])):
    decoded = tokenizer.decode(sample['input_ids'][i], skip_special_tokens=False)
    print(decoded)

In [ ]:
# TRAINING

print("FINE-TUNING")
print(f"Model: {model_name}")
print(f"Method: LoRA (r={lora_config.r}, alpha={lora_config.lora_alpha})")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size} x {training_args.gradient_accumulation_steps}")
print(f"Learning rate: {training_args.learning_rate}\n")

# Start training
trainer.train()

print("TRAINING COMPLETED")

In [ ]:
# SAVE FINE-TUNED MODEL

output_dir = f"models/fine_tuned_model/train{train_value}"
# Save the LoRA adapter weights and tokenizer
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Model saved to {output_dir}")
print("\nSaved files:")
print("  - adapter_config.json (LoRA configuration)")
print("  - adapter_model.bin (LoRA weights)")
print("  - tokenizer files")
print("\nTo load later:")
print("  from peft import PeftModel")
print("  model = AutoModelForCausalLM.from_pretrained('mistralai/Mistral-7B-Instruct-v0.2')")
print(f"  model = PeftModel.from_pretrained(model, '{output_dir}')")